# Yambda AutoFE — запуск в Google Colab

GPU не нужен (Runtime → Change runtime type → CPU). Результаты сохраняются на Google Drive после каждого эксперимента, чтобы не потеряться при отключении сессии. Держите вкладку открытой.

**Память:** бесплатный Colab ≈ 12 ГБ RAM → основной прогон на 5000 пользователей. При High-RAM (Colab Pro) можно поставить `N_USERS = None` (все 10k).

In [ ]:
N_USERS = 5000   # None = все 10k пользователей (нужно >= 25 ГБ RAM)

from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/yambda-autofe-results'
import os; os.makedirs(OUT, exist_ok=True)
!git clone -q https://github.com/georgiken/yambda-autofe.git /content/recsys
%cd /content/recsys
!pip install -q -r requirements.txt 2>&1 | tail -2
!nproc; free -g

In [ ]:
# 1) тесты на утечки (~15 с)
!python -m pytest -q tests

In [ ]:
# 2) основной эксперимент
main = 'u10000' if N_USERS is None else f'u{N_USERS}'
flag = '' if N_USERS is None else f'--n-users {N_USERS}'
!python scripts/run_experiments.py {flag} --out {OUT}/{main} 2>&1 | grep -v Warning

In [ ]:
# 3) масштабируемость: 1k пользователей (полные истории)
!python scripts/run_experiments.py --n-users 1000 --out {OUT}/u1000 2>&1 | grep -v Warning

In [ ]:
# 4) таблицы и графики
!rm -f {OUT}/events_*.parquet
!python scripts/make_report.py {OUT}
!cd {OUT} && zip -qr /content/results.zip . && ls -la /content/results.zip